In [1]:
import pandas as pd
import altair as alt
import json
from IPython.display import HTML

alt.data_transformers.enable("default", max_rows=None)
alt.renderers.enable("mimetype")

btc_h = pd.read_csv("../data/processed/btc_history_processed.csv", parse_dates=["date"])
cg    = pd.read_csv("../data/processed/coingecko_processed.csv",   parse_dates=["date"])
wb    = pd.read_csv("../data/processed/world_bank_processed.csv")

btc_h = btc_h.dropna(subset=["price_change_pct"])

print("BTC history:", btc_h.date.min().date(), "→", btc_h.date.max().date(), f"({len(btc_h)} rows)")
print("CoinGecko coins:", cg.coin.unique().tolist())
print("World Bank countries:", wb.country.nunique(), "| years:", wb.year.min(), "–", wb.year.max())

BTC history: 2024-02-20 → 2026-02-18 (730 rows)
CoinGecko coins: ['binancecoin', 'bitcoin', 'ethereum', 'solana', 'tether']
World Bank countries: 15 | years: 2018 – 2023


In [2]:
# ── Chart 1: Bitcoin Daily Return Calendar Heatmap ─────────────────────────────
btc_h["day"]       = btc_h["date"].dt.day
btc_h["month_str"] = btc_h["date"].dt.strftime("%Y-%m")
month_order = sorted(btc_h["month_str"].unique().tolist())

calendar = alt.Chart(btc_h).mark_rect(cornerRadius=2).encode(
    x=alt.X("day:O", title="Day of Month"),
    y=alt.Y("month_str:O", title="", sort=month_order),
    color=alt.Color(
        "price_change_pct:Q",
        scale=alt.Scale(scheme="redyellowgreen", domain=[-10, 10], clamp=True),
        title="Return (%)"
    ),
    tooltip=[
        alt.Tooltip("date:T", title="Date"),
        alt.Tooltip("price_change_pct:Q", title="Daily Return (%)", format=".2f"),
        alt.Tooltip("market_price_usd:Q", title="BTC Price (USD)", format="$,.0f"),
    ]
).properties(
    width=700, height=370,
    title=alt.TitleParams(
        "Bitcoin Daily Return Calendar (2024–2026)",
        subtitle=["Green = positive day | Red = negative day | Hover for details"],
        color="white", subtitleColor="#aaa", fontSize=16
    )
).configure(
    background="#1e1e1e"
).configure_axis(
    labelColor="white", titleColor="white", gridColor="#333"
).configure_legend(
    labelColor="white", titleColor="white"
)

calendar

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [3]:
# ── Chart 2: Whale Activity Scatter ────────────────────────────────────────────
# avg_tx_value_usd = avg USD per on-chain transaction → high value = fewer big movers (whale signal)

scatter = alt.Chart(btc_h).mark_circle(opacity=0.72).encode(
    x=alt.X(
        "avg_tx_value_usd:Q",
        title="Avg Transaction Value (USD) — Whale Proxy  [log scale]",
        scale=alt.Scale(type="log"),
        axis=alt.Axis(format="$~s", labelColor="white", titleColor="white")
    ),
    y=alt.Y(
        "price_change_pct:Q",
        title="Bitcoin Daily Price Change (%)",
        axis=alt.Axis(labelColor="white", titleColor="white")
    ),
    size=alt.Size(
        "tx_count:Q",
        title="Tx Count",
        scale=alt.Scale(range=[25, 450]),
        legend=alt.Legend(labelColor="white", titleColor="white")
    ),
    color=alt.Color(
        "price_change_pct:Q",
        scale=alt.Scale(scheme="redyellowgreen", domain=[-10, 10], clamp=True),
        legend=None
    ),
    tooltip=[
        alt.Tooltip("date:T",              title="Date"),
        alt.Tooltip("avg_tx_value_usd:Q",  title="Avg Tx Value (USD)", format="$,.0f"),
        alt.Tooltip("price_change_pct:Q",  title="Price Change (%)",   format=".2f"),
        alt.Tooltip("tx_count:Q",          title="Transaction Count",  format=","),
        alt.Tooltip("hash_rate:Q",         title="Hash Rate",          format=",.0f"),
    ]
).properties(
    width=720, height=430,
    title=alt.TitleParams(
        "Whale Activity vs Bitcoin Daily Price Change",
        subtitle=[
            "Each point = 1 trading day  |  Size = # transactions  |  Color = daily return",
            "High avg tx value → fewer, larger on-chain transfers → whale signal"
        ],
        color="white", subtitleColor="#aaa", fontSize=16
    )
).interactive().configure(
    background="#1e1e1e"
).configure_axis(
    labelColor="white", titleColor="white", gridColor="#2d2d2d"
)

scatter

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [4]:
# ── Chart 3: Annotated Timeline — BTC Price + Whale Signal + Global Events ─────
events = pd.DataFrame([
    {"date": "2024-03-14", "label": "BTC ATH $73k",    "row": 0},
    {"date": "2024-04-20", "label": "Halving",          "row": 1},
    {"date": "2024-08-05", "label": "Flash Crash",      "row": 0},
    {"date": "2024-11-05", "label": "Trump Elected",    "row": 1},
    {"date": "2024-12-17", "label": "ATH $106k",        "row": 0},
    {"date": "2025-01-20", "label": "Inauguration",     "row": 1},
    {"date": "2025-02-03", "label": "Tariff Shock",     "row": 0},
    {"date": "2025-04-07", "label": "Trade War Crash",  "row": 1},
])
events["date"] = pd.to_datetime(events["date"])

# Filter events to our data range
events = events[(events["date"] >= btc_h["date"].min()) & (events["date"] <= btc_h["date"].max())]

# ── Top panel: BTC price + event annotations
price_line = alt.Chart(btc_h).mark_line(color="#f7931a", strokeWidth=2).encode(
    x=alt.X("date:T", title=None),
    y=alt.Y("market_price_usd:Q", title="BTC Price (USD)", axis=alt.Axis(format="$~s")),
    tooltip=[alt.Tooltip("date:T"), alt.Tooltip("market_price_usd:Q", title="Price", format="$,.0f")]
)

event_rules = alt.Chart(events).mark_rule(
    color="white", opacity=0.3, strokeDash=[5, 4], strokeWidth=1.5
).encode(x="date:T")

event_labels = alt.Chart(events).mark_text(
    color="white", fontSize=9.5, align="center", fontWeight="bold"
).encode(
    x="date:T",
    y=alt.Y("row:O", scale=alt.Scale(domain=[0, 1], range=[28, 52])),
    text="label:N"
)

price_panel = (price_line + event_rules + event_labels).properties(
    width=780, height=260,
    title=alt.TitleParams("Bitcoin Price + Global Event Markers", color="white", fontSize=14)
)

# ── Bottom panel: Whale signal (avg tx value)
whale_area = alt.Chart(btc_h).mark_area(
    color="#9c27b0", opacity=0.45,
    line={"color": "#ce93d8", "strokeWidth": 1.5}
).encode(
    x=alt.X("date:T", title="Date"),
    y=alt.Y("avg_tx_value_usd:Q", title="Avg Tx Value (USD)", axis=alt.Axis(format="$~s")),
    tooltip=[
        alt.Tooltip("date:T"),
        alt.Tooltip("avg_tx_value_usd:Q", title="Avg Tx Value", format="$,.0f")
    ]
)

whale_panel = (whale_area + event_rules).properties(
    width=780, height=160,
    title=alt.TitleParams(
        "On-Chain Avg Transaction Value (Whale Activity Proxy)",
        color="#ce93d8", fontSize=14
    )
)

timeline = alt.vconcat(price_panel, whale_panel, spacing=12).configure(
    background="#1e1e1e"
).configure_axis(
    labelColor="white", titleColor="white", gridColor="#2a2a2a"
).configure_legend(
    labelColor="white", titleColor="white"
)

timeline

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c

<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [5]:
# ── Chart 4: Monthly Volatility Heatmap — Bitcoin vs Ethereum ──────────────────
btc_eth = cg[cg["coin"].isin(["bitcoin", "ethereum"])].copy()
btc_eth["month_str"] = btc_eth["date"].dt.strftime("%Y-%m")
month_order_cg = sorted(btc_eth["month_str"].unique().tolist())

monthly = btc_eth.groupby(["coin", "month_str"])["daily_return"].agg(
    volatility="std",
    avg_return="mean"
).reset_index()

coin_labels = {"bitcoin": "Bitcoin", "ethereum": "Ethereum"}
monthly["coin_label"] = monthly["coin"].map(coin_labels)

vol_heatmap = alt.Chart(monthly).mark_rect(cornerRadius=3, stroke="#1e1e1e", strokeWidth=1).encode(
    x=alt.X("month_str:O", title="Month", sort=month_order_cg,
            axis=alt.Axis(labelAngle=-45, labelColor="white", titleColor="white")),
    y=alt.Y("coin_label:N", title="",
            axis=alt.Axis(labelColor="white", titleColor="white")),
    color=alt.Color(
        "volatility:Q",
        scale=alt.Scale(scheme="orangered"),
        title="Volatility\n(σ of daily %)",
        legend=alt.Legend(labelColor="white", titleColor="white")
    ),
    tooltip=[
        alt.Tooltip("coin_label:N",  title="Coin"),
        alt.Tooltip("month_str:O",   title="Month"),
        alt.Tooltip("volatility:Q",  title="Volatility (std)", format=".3f"),
        alt.Tooltip("avg_return:Q",  title="Avg Return (%)",   format=".2f"),
    ]
).properties(
    width=650, height=130,
    title=alt.TitleParams(
        "Monthly Volatility: Bitcoin vs Ethereum (Feb 2025 – Feb 2026)",
        subtitle=["Darker orange = more volatile month  |  Do global events spike both coins together?"],
        color="white", subtitleColor="#aaa", fontSize=15
    )
).configure(
    background="#1e1e1e"
).configure_axis(
    labelColor="white", titleColor="white", gridColor="#333"
)

vol_heatmap

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [6]:
# ── Chart 5: Altair World Map — Economic Stress & Crypto Exposure ───────────────
CENTROIDS = {
    "ARG": [-63.6, -38.4], "BRA": [-51.9, -14.2], "CHN": [104.2,  35.9],
    "SLV": [-88.9,  13.8], "IND": [ 78.9,  20.6], "KEN": [ 36.8,  -1.3],
    "NGA": [  8.7,   9.1], "PHL": [121.8,  12.9], "RUS": [105.3,  61.5],
    "ZAF": [ 22.9, -30.6], "TUR": [ 35.2,  38.9], "UKR": [ 31.2,  48.4],
    "USA": [-95.7,  37.1], "VEN": [-66.6,   6.4], "VNM": [108.3,  14.1],
}

wb_avg = wb.groupby(["country", "iso_code"]).agg(
    avg_inflation  = ("inflation",  "mean"),
    avg_gdp_growth = ("gdp_growth", "mean"),
    avg_remittance = ("remittance", "mean"),
).reset_index()

wb_avg["lon"]            = wb_avg["iso_code"].map(lambda c: CENTROIDS.get(c, [None, None])[0])
wb_avg["lat"]            = wb_avg["iso_code"].map(lambda c: CENTROIDS.get(c, [None, None])[1])
wb_avg["remittance_pct"] = (wb_avg["avg_remittance"] * 100).round(2)
wb_avg = wb_avg.dropna(subset=["lon", "lat"])

world_topo = alt.topo_feature(
    "https://cdn.jsdelivr.net/npm/world-atlas@2/countries-110m.json",
    "countries"
)

background = alt.Chart(world_topo).mark_geoshape(
    fill="#252525",
    stroke="#3a3a3a",
    strokeWidth=0.4
)

bubbles = alt.Chart(wb_avg).mark_point(filled=True, opacity=0.82).encode(
    longitude="lon:Q",
    latitude="lat:Q",
    size=alt.Size("remittance_pct:Q",
                  scale=alt.Scale(range=[80, 2200]),
                  legend=alt.Legend(title="Remittance (% GDP)", orient="bottom-right",
                                    labelColor="white", titleColor="white")),
    color=alt.Color("avg_inflation:Q",
                    scale=alt.Scale(scheme="orangered"),
                    legend=alt.Legend(title="Avg Inflation (%)", orient="bottom-left",
                                     labelColor="white", titleColor="white")),
    stroke=alt.value("#64b5f6"),
    strokeWidth=alt.value(1.8),
    tooltip=[
        alt.Tooltip("country:N",         title="Country"),
        alt.Tooltip("avg_inflation:Q",   title="Avg Inflation (%)",    format=".1f"),
        alt.Tooltip("avg_gdp_growth:Q",  title="Avg GDP Growth (%)",   format=".1f"),
        alt.Tooltip("remittance_pct:Q",  title="Remittance (% GDP)",   format=".1f"),
    ]
)

(background + bubbles).project(
    type="naturalEarth1"
).properties(
    width=880, height=460,
    title=alt.TitleParams(
        "Global Economic Stress & Crypto Exposure (2018–2023 Avg)",
        subtitle=["Circle size = Remittance flows (% GDP)  |  Fill = Avg Inflation  |  Hover for details"],
        color="white", subtitleColor="#aaa", fontSize=16, subtitleFontSize=11
    )
).configure(
    background="#1e1e1e"
).configure_view(
    strokeOpacity=0
).configure_title(
    color="white", subtitleColor="#aaa"
)

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting
